# Decision Theory in Binary Classification

**Initial Due Date: 2026-01-15 10:00AM**  
**Final Due Date: 2026-01-19 4:15PM**

## Learning Objectives

By the end of this activity, you will be able to:

-   Construct ROC curves describing the tradeoff between error rates in
    binary classification.
-   Compute the Area Under the Curve (AUC) for ROC curves.
-   Determine the optimal decision threshold for a binary classifier
    given asymmetrical costs of different types of classification
    errors.

## Introduction

We [recently
introduced](https://middcs.github.io/data-science-notes/source/25-classification.html)
classification, the task of predicting a discrete label from input
features. We focused on logistic regression for predicting binary
(“yes/no” or 1/0 labels). Logistic regression is one of several
*score-based linear models*, which produces a prediction by computing a
*score*

$$
\begin{aligned}
    s = \sum_{i = 1}^p w_i x_i
\end{aligned}
$$

and then predicting a label of 1 if $s$ exceeds some threshold $t$ and 0
otherwise. In this activity, we’ll explore the impact of varying the
threshold $t$ on the performance of a score-based binary classifier like
logistic regression. We’ll first consider this question from the
standpoint of error rates, which will lead us to the concept of the
*Receiver Operating Characteristic (ROC) curve*. Then, we’ll consider it
from the standpoint of decision-theory, in which the cost of being wrong
is not the same for all types of errors.

## Part A: Train a Model

### Exercise A1

First, download the Australia weather data set, prepare it for binary
classification, split it into training and test sets, and train a
logistic regression model to predict whether it will rain tomorrow.
Please name your model `model`. We encourage you to reuse code from the
[class
notes](https://middcs.github.io/data-science-notes/source/25-classification.html)
for this exercise.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

sns.set(style="whitegrid")

url = "https://raw.githubusercontent.com/middcs/data-science-notes/refs/heads/main/data/australia-weather/weatherAUS.csv"

# TODO: Your code here

### Exercise A2

We can extract the scores for the test set using the `predict_proba`
method of the trained model. `predict_proba` returns a 2-column array,
where the first column gives the predicted probability of label 0 and
the second column gives the predicted probability of label 1. Please
extract these scores and select just the second column, which describes
the predicted probability of label 1. We can think of these as
normalized scores for the positive class, which are guaranteed to lie
between 0 and 1. ***Assign this array to a variable named `y_scores`.***

> #### ❓ Why don’t we need both columns?
>
> We don’t need both columns because the row must sum to 1, so knowing
> one of them tells us the other.

In [ ]:
# TODO: Your code here
y_scores

### Exercise A3

Recall that our decision rule is to predict label 1 if the score exceeds
some threshold $t$ and label 0 otherwise. Write a function
`predict_with_threshold` that takes as input an array of scores and a
threshold value, and returns an integer array of predicted labels (0 or
1). Recall that when booleans are converted to integers in Python,
`True` becomes 1 and `False` becomes 0.

In [ ]:
# TODO: Your code here

You can check your implementation by comparing the predictions from your
function with those from the model when using a threshold of 0.5 (the
default for the model’s `predict` method). If the cell below returns
`np.True_` then you have passed this check and your implementation is
likely correct.

In [ ]:
np.all(predict_with_threshold(y_scores, 0.5) == model.predict(X_test))

### Exercise A4

Now write a function named `positive_rates` with 3 arguments, a
`np.ndarray` of thresholds, the score array, and the array of true
labels, and returns a tuple with two arrays: the true positive rate and
the false positive rate of the classifier for each threshold. So, if you
pass the array:

``` python
t = np.linspace(0, 1, 5)
```

to

``` python
# Python note: Here during assignment we "unpack" tuples of known length into
# individual variables.
tp_rates, fp_rates = positive_rates(t, y_scores, y_test)
```

you should obtain arrays `tp_rates` and `fp_rates`, each of length 5,
where `tp_rates[i]` is the true positive rate and `fp_rates[i]` is the
false positive rate when using threshold `t[i]`.

***Note***: While it is possible to use functions from `scikit-learn` to
compute true positive rates and false positive rates, please implement
this functionality from the definitions using only NumPy or Pandas
operations.

Suggestions:

-   You are likely need a for-loop over the thresholds.
-   You can use your `predict_with_threshold` function to get the
    predicted labels for each threshold. Then, you can compute true
    positives, false negatives, false positives, and true negatives by
    comparing the predicted labels to the true labels.
-   Then, you can compute the true positive rate and false positive rate
    using their definitions. If the denominator in either rate is zero,
    we define the rate to be 0.

In [ ]:
# TODO: Your code here

## Part B: ROC Curve

For a given threshold, we now have a way to compute the true positive
rate and false positive rate of our classifier. There’s not necessarily
one “best” threshold to use for all applications. Rather than trying to
choose a “best” threshold right now, we can instead map all our possible
choices using the Receiver Operating Characteristic (ROC) curve for our
classifier.

### Exercise B1

Once you’ve implemented `positive_rates`, use it to compute the true
positive rates and false positive rates for thresholds ranging from 0 to
1 in increments of 0.01. Then, make a lineplot in which the horizontal
axis is the false positive rate and the vertical axis is the true
positive rate.

In [ ]:
# TODO: Your code here
plt.show()

This plot is called the Receiver Operating Characteristic (ROC) curve
for our binary classifier on this data set. The ROC curve summarizes the
performance of our classifier across possible thresholds, and indicates
the tradeoff between true positive rate and false positive rate. For
example, the ROC curve says that if we are willing to accept a false
positive rate of about `python fp_rates[50]` then we can achieve a true
positive rate of about `python tp_rates[50]`. The closer the ROC curve
comes to the top-left corner of the plot, the stronger the overall
performance of our classifier. In contrast a ROC curve that is close to
the diagonal line from (0,0) to (1,1) indicates a classifier that is not
much better than random guessing.

### Exercise B2

The Area Under The Curve (AUC) is a common measure for the overall
quality of a classifier. The AUC is literally what its name suggests:
the area under the ROC curve, in the sense of an integral in Calculus.
An ROC that achieves perfect classification (100% true positive rate and
0% false positive rate) has an AUC of 1.0, while a classifier that makes
random predictions has an expected AUC of 0.5.

Implement a function name `roc_auc` that has two arguments, the true
positive rates and false positive rates, and returns the AUC. Then, use
your function to compute the AUC for the ROC curve you generated above,
recalling that an AUC close to 1.0 indicates a strong classifier. You
can use the trapezoid rule to approximate the area under the curve:

$$
\begin{aligned}
    \mathrm{Area} \approx \frac{1}{2}\sum_{i = 1}^{n-1} (\mathrm{TPR}(t_{i-1}) +  \mathrm{TPR}(t_{i})) \cdot \left(\mathrm{FPR}(t_{i}) - \mathrm{FPR}(t_{i-1})\right)
\end{aligned}
$$

Suggestions:

-   It is possible to evaluate this sum with a for-loop. In the spirit
    of this course, however, **use Numpy operations with no explicit
    for-loops** to compute the sum efficiently.
-   Recall that you can use array slicing to select portions of an
    array. A slice is defined using the syntax
    `array[start:stop(:step)]`, where `start` is the inclusive start
    index, `stop` is the exclusive end index, and `step` is the spacing
    between elements (default 1). These indices can also be negative, in
    which you count from the end (where -1 is the index of the last
    element). For example, `array[1:]` selects all elements except the
    first, and `array[:-1]` selects all elements except the last.

In [ ]:
# TODO: Your code here
auc = roc_auc(tp_rates, fp_rates)
print(f"AUC: {auc:.4f}")

## Part C: Decision Theory in Classification

In many classification applications, the costs of different types of
errors are not the same. For example, suppose you are going to use our
rain predictor to decide whether or not to bring an umbrella with you
when you leave today. There are four possible outcomes:

-   **True positive (TP)**: You predicted rain and brought your
    umbrella, and it did indeed rain! **You are dry and happy.**
-   **True negative (TN)**: You predicted no rain and did not bring your
    umbrella, and it did not rain. **You are dry and happy.**
-   **False positive (FP)**: You predicted rain and brought your
    umbrella, but it did not rain. You are dry, but you had to carry
    around an unnecessary umbrella all day and your friends made fun of
    you. **You are dry but lightly embarrassed.**  
-   **False negative (FN)**: You predicted no rain and did not bring
    your umbrella, but it rained. **You are wet and very embarrassed.**

When modeling these outcomes, we might reasonably think that the **TP**
and **TN** outcomes are both “good,” the **FP** outcome is “somewhat
bad,” and the **FN** outcome is “very bad.” To formalize this, we can
assign *utilities* to each outcome. For example, we might assign
utilities as follows:

| Outcome | Utility |
|---------|---------|
| TP      | 0       |
| TN      | 0       |
| FP      | -1      |
| FN      | -5      |

This scenario reflects *asymmetrical cost of error*: we want to avoid
false negatives much more than false positives.

Given these utilities, we can compute the *expected utility* of our
classifier at a given threshold $t$ as follows:

$$
\begin{aligned}
    \mathrm{EU}(t) = \frac{1}{n} \mathrm{FP}(t) \cdot U_{FP} + \frac{1}{n} \mathrm{FN}(t) \cdot U_{FN}\;,
\end{aligned}
$$

where $\mathrm{FP}(t)$ is the number of false positives at threshold
$t$, $\mathrm{FN}(t)$ is the number of false negatives at threshold $t$,
$U_{FP}$ is the utility of a false positive, $U_{FN}$ is the utility of
a false negative, and $n$ is the total number of predictions. We do not
include true positives or true negatives in this calculation, since
their utility is zero.

## Exercise C1

Implement the function below, named `expected_utility`, which returns
the expected utility of our classifier at each threshold in a given
array of thresholds, using the utilities that the user can pass in as
optional arguments. Your implementation will likely be quite similar to
your implementation of `positive_rates`.

In [ ]:
def expected_utility(thresholds, y_scores, y_true, U_FP, U_FN):
    # TODO: Your code here
    return 0  # Replace with your implementation

## Exercise C2

Now, use this function to compute the expected utility at each threshold
from 0 to 1 in increments of 0.01, and make a line plot of expected
utility versus threshold. Your plot should have two lines:

1.  Using the utilities $U_{FP} = -1$ and $U_{FN} = -5$ as in the table
    above.
2.  Using the utilities $U_{FP} = -1$ and $U_{FN} = -2$, which reflects
    a scenario in which false negatives are still worse than false
    positives but not as dramatically so.

These two utility combinations might be used by a person who hates
getting wet (first case) and one who isn’t too bothered by getting a
little wet (second case). Recall that `sns.lineplot` has an optional
`ax` axis argument that allows you to plot another line on an existing
axis.

In [ ]:
# TODO: Your code here
plt.show()

### Exercise C3

Determine the optimal threshold for each of the two utility scenarios by
finding the threshold that maximizes expected utility. You can do this
by finding the index of the maximum expected utility value in each case,
and then using that index to look up the corresponding threshold value
(check out
[`np.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html)).
Assign these optimal thresholds to variables named `best_threshold_1`
(for $U_{FN }= -5$ and $U_{FP} = -1$) and `best_threshold_2` (for
$U_{FN} = -2$ and $U_{FP} = -1$), and print them out.

In [ ]:
# TODO: Your code here

## Collaboration statement

In a new text cell immediately below this paragraph (or by editing this
text cell to add a paragraph), briefly list who or what you collaborated
with and how. Cite any sources here or with relevant inline comments in
your code. Acknowledge all contributors, both people and AI, and what
portions of this notebook they contributed. You do not need to cite or
acknowledge any material provided in the starter file(s).

## Submitting your notebook

You will simultaneously submit the following two files to the relevant
assignment on [Gradescope](https://gradescope.com) via the “Upload
option” (guide
[here](https://guides.gradescope.com/hc/en-us/articles/21865616724749-Submitting-a-Code-assignment)).
**Both files must be uploaded at the same time and the file names must
match the specification exactly for the autotesting to run
successfully.**

1.  `activity_ROC_curve.ipynb`: Your completed IPython notebook. You can
    obtain this via the “File→Download→Download .ipynb” menu option in
    Colab.
2.  `activity_ROC_curve.py`: Your completed IPython notebook as a Python
    file. You can obtain this via the “File→Download→Download .py” menu
    option in Colab. This file is used to provide line-level feedback on
    your submission.

You can submit multiple times, with only the most recent submission
(before the final due date) assessed for credit. Gradescope will run a
series of automated unit tests on your notebook (which may takes 10s of
seconds depending on the complexity of the notebook). Note that the
tests performed by Gradescope are limited. Passing all of the visible
tests does not guarantee that your submission correctly satisfies all of
the requirements of the assignment.